## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are Pairing of:
1. Schema, including the name, description, and parameters of the tool.

2. A function or coroutine to execute the tool.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
)

In [3]:
response = model.invoke("Write a poem about the beauty of coding.")
response

AIMessage(content="In realms of ones and zeroes bright,\nA world of wonder, day and night.\nWhere lines of code, like rivers flow,\nCreating worlds, for all to know.\n\nThe beauty lies, in every line,\nA symphony of logic and design.\nA dance of bits, a harmony of thought,\nAs algorithms weave, a tapestry brought.\n\nThe programmer's art, a canvas wide,\nWhere creativity and problem-solving reside.\nA challenge met, with each new test,\nA solution found, and a new quest.\n\nIn this digital realm, where code reigns,\nA beauty shines, in every pixel's frame.\nA world of possibility, where dreams unfold,\nA place where imagination, is the only gold.\n\nThe beauty of coding, is not just in the code,\nBut in the joy, of creating something new to hold.\nA sense of pride, in every line of sight,\nA knowledge gained, in the dark of night.\n\nSo let us celebrate, this digital art,\nWhere code and beauty, forever will start.\nFor in the world of coding, we find our way,\nTo a realm of wonder, wh

In [4]:
from langchain.tools import tool


@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is sunny."


model_with_tools = model.bind_tools([get_current_weather])

In [5]:
response = model_with_tools.invoke("What is the current weather in New York?")

In [6]:
print(response)

content='' additional_kwargs={'tool_calls': [{'id': '09kxwcyav', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 227, 'total_tokens': 243, 'completion_time': 0.024087971, 'completion_tokens_details': None, 'prompt_time': 0.014942308, 'prompt_tokens_details': None, 'queue_time': 0.15888266, 'total_time': 0.039030279}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f0f07-339d-70a1-a8f2-643bdfc55ea5-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': '09kxwcyav', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 227, 'output_tokens': 16, 'total_tokens': 243}


In [7]:
response.tool_calls

[{'name': 'get_current_weather',
  'args': {'location': 'New York'},
  'id': '09kxwcyav',
  'type': 'tool_call'}]

### Tool Execution Loop

In [ ]:
# Step 1: Model generates a tool calls
messages = [
    {
        "role": "user",
        "content": "What is the current weather in New York?"
    }
]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute the tool calls and get the results
for tool_call in ai_msg.tool_calls:
    args = tool_call["args"] if isinstance(tool_call, dict) else tool_call.args
    tool_result = get_current_weather.invoke(args)
    messages.append(tool_result)

# Step 3: Model generates a final response based on the tool results
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It seems the current weather in New York is sunny.


In [20]:
messages

[{'role': 'user', 'content': 'What is the current weather in New York?'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hv3smpnc6', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 227, 'total_tokens': 243, 'completion_time': 0.024920641, 'completion_tokens_details': None, 'prompt_time': 0.023330065, 'prompt_tokens_details': None, 'queue_time': 0.05270723, 'total_time': 0.048250706}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f0f0d-186d-7783-9ace-9cd098d3bade-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'hv3smpnc6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 227, 'output_tokens': 16, 'total_tokens': 243}),

In [21]:
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020F77C08E90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020F78D38BD0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_current_weather', 'description': 'Get the 